In [245]:
# import necessary libraries
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer



In [246]:
# read the dataset
df=pd.read_csv("Dataset.csv")
df.head(30)

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,United Kingdom,9/24/2021,2021,TV-14,9 Seasons,"British TV Shows, Reality TV"
6,s10,Movie,The Starling,Theodore Melfi,United States,9/24/2021,2021,PG-13,104 min,"Comedies, Dramas"
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,India,5/1/2021,2019,TV-Y7,87 min,"Children & Family Movies, Comedies, Music & Mu..."
8,s13,Movie,Je Suis Karl,Christian Schwochow,Germany,9/23/2021,2021,TV-MA,127 min,"Dramas, International Movies"
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,India,5/1/2021,2013,TV-Y7,76 min,"Children & Family Movies, Music & Musicals"


In [247]:
# check the number of rows and columns in the dataset
df.shape

(8790, 10)

In [248]:
# check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [249]:
# check for missing values
df.isnull().sum()

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

In [250]:
# check for duplicates
df.duplicated().sum()

np.int64(0)

In [251]:
# drop columns that are not needed for analysis
df.drop(['show_id',"rating" ], axis=1, inplace=True)


In [252]:
# extract the yaer from the date 
df['date_added'] = pd.to_datetime(
    df['date_added'],
).dt.year

In [253]:
df['date_added'].head()

0    2021
1    2021
2    2021
3    2021
4    2021
Name: date_added, dtype: int32

In [254]:
df.head()

,type,title,director,country,date_added,release_year,duration,listed_in
0,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021,2020,90 min,Documentaries
1,TV Show,Ganglands,Julien Leclercq,France,2021,2021,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,TV Show,Midnight Mass,Mike Flanagan,United States,2021,2021,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021,2021,91 min,"Children & Family Movies, Comedies"
4,Movie,Sankofa,Haile Gerima,United States,2021,1993,125 min,"Dramas, Independent Movies, International Movies"


In [255]:
# check the listed_in column and check nulll columns
print(df["listed_in"].head())
print(df["listed_in"].isna().sum())

0                                        Documentaries
1    Crime TV Shows, International TV Shows, TV Act...
2                   TV Dramas, TV Horror, TV Mysteries
3                   Children & Family Movies, Comedies
4     Dramas, Independent Movies, International Movies
Name: listed_in, dtype: object
0


In [256]:
#split the listed_in column
# df["genre_list"]=df["listed_in"].str.split(",")
df['genre_list'] = df['listed_in'].str.split(',').apply(lambda lst: [g.strip() for g in lst])

In [257]:
print(df[['listed_in', 'genre_list']].head())
print(df['genre_list'].isna().sum())

                                           listed_in  \
0                                      Documentaries   
1  Crime TV Shows, International TV Shows, TV Act...   
2                 TV Dramas, TV Horror, TV Mysteries   
3                 Children & Family Movies, Comedies   
4   Dramas, Independent Movies, International Movies   

                                          genre_list  
0                                    [Documentaries]  
1  [Crime TV Shows, International TV Shows, TV Ac...  
2               [TV Dramas, TV Horror, TV Mysteries]  
3               [Children & Family Movies, Comedies]  
4  [Dramas, Independent Movies, International Mov...  
0


In [258]:
# count the unique values
all_genres = df['genre_list'].explode()
print(all_genres.value_counts())

genre_list
International Movies            2752
Dramas                          2426
Comedies                        1674
International TV Shows          1349
Documentaries                    869
Action & Adventure               859
TV Dramas                        762
Independent Movies               756
Children & Family Movies         641
Romantic Movies                  616
Thrillers                        577
TV Comedies                      573
Crime TV Shows                   469
Kids' TV                         448
Docuseries                       394
Music & Musicals                 375
Romantic TV Shows                370
Horror Movies                    357
Stand-Up Comedy                  343
Reality TV                       255
British TV Shows                 252
Sci-Fi & Fantasy                 243
Sports Movies                    219
Anime Series                     174
Spanish-Language TV Shows        173
TV Action & Adventure            167
Korean TV Shows            

In [259]:
# check the unique columns
print(all_genres.nunique())


42


In [260]:
# extract duration information
df['minutes'] = df['duration'].str.extract(r'(\d+) min')[0].astype(float).fillna(0)
df['season'] = df['duration'].str.extract(r'(\d+) Season')[0].astype(float).fillna(0)

In [261]:
# check the new columns
print(df[['duration', 'minutes', 'season']].head(10))

    duration  minutes  season
0     90 min     90.0     0.0
1   1 Season      0.0     1.0
2   1 Season      0.0     1.0
3     91 min     91.0     0.0
4    125 min    125.0     0.0
5  9 Seasons      0.0     9.0
6    104 min    104.0     0.0
7     87 min     87.0     0.0
8    127 min    127.0     0.0
9     76 min     76.0     0.0


In [262]:
# change the columns to integer type
df['minutes'] = df['minutes'].astype(int)
df['season'] = df['season'].astype(int)

In [263]:
# check the new columns
df[['duration', 'minutes', 'season']].head()

,duration,minutes,season
0,90 min,90,0
1,1 Season,0,1
2,1 Season,0,1
3,91 min,91,0
4,125 min,125,0


In [264]:
# drop the original duration column
df.drop(['duration'], axis=1, inplace=True)

In [265]:
df.head()

,type,title,director,country,date_added,release_year,listed_in,genre_list,minutes,season
0,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021,2020,Documentaries,[Documentaries],90,0
1,TV Show,Ganglands,Julien Leclercq,France,2021,2021,"Crime TV Shows, International TV Shows, TV Act...","[Crime TV Shows, International TV Shows, TV Ac...",0,1
2,TV Show,Midnight Mass,Mike Flanagan,United States,2021,2021,"TV Dramas, TV Horror, TV Mysteries","[TV Dramas, TV Horror, TV Mysteries]",0,1
3,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021,2021,"Children & Family Movies, Comedies","[Children & Family Movies, Comedies]",91,0
4,Movie,Sankofa,Haile Gerima,United States,2021,1993,"Dramas, Independent Movies, International Movies","[Dramas, Independent Movies, International Mov...",125,0


In [266]:
# encode the genre_list
mlb = MultiLabelBinarizer()
genre_matrix = pd.DataFrame(
    mlb.fit_transform(df['genre_list']),
    columns=mlb.classes_,
    index=df.index
).add_prefix('genre_')

df = pd.concat([df, genre_matrix], axis=1)
print(df.shape)


(8790, 52)


In [267]:
# drop the listed_in and genre_list columns.
df=df.drop(columns=["listed_in","genre_list"])

In [268]:
df.head()

,type,title,director,country,date_added,release_year,minutes,season,genre_Action & Adventure,genre_Anime Features,...,genre_TV Action & Adventure,genre_TV Comedies,genre_TV Dramas,genre_TV Horror,genre_TV Mysteries,genre_TV Sci-Fi & Fantasy,genre_TV Shows,genre_TV Thrillers,genre_Teen TV Shows,genre_Thrillers
0,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021,2020,90,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,TV Show,Ganglands,Julien Leclercq,France,2021,2021,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
2,TV Show,Midnight Mass,Mike Flanagan,United States,2021,2021,0,1,0,0,...,0,0,1,1,1,0,0,0,0,0
3,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021,2021,91,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Movie,Sankofa,Haile Gerima,United States,2021,1993,125,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [269]:
df.to_csv("netflix_clean.csv", index=False)